# Duplicate Image Remover

Removes duplicate images from the dataset using the perceptual hash column produced by `1.dataset_stats.ipynb`. Within each group of images sharing the same hash, one representative is kept according to the following priority rules:

1. Largest file size (`size_mb`)
2. Highest pixel resolution (`width x height`) if sizes are tied
3. Known capture device (non-`unknown`) if resolutions are tied
4. First image in the group as a final tiebreaker

Duplicate images are moved to a configurable destination folder rather than deleted, so the operation is fully reversible.

**Input:** `duplicates.csv` (produced by notebook 1)  
**Output:** duplicate images moved to `destination_folder`


## 1. Import Libraries

In [14]:
import pandas as pd
import os
import shutil
from IPython.display import display

## 2. Load and Explore Data

In [ ]:
df = pd.read_csv("duplicates.csv")

print(f"Total duplicate records: {len(df)}")
print(f"Unique image hashes: {df['image_hash'].nunique()}")
print(f"\nFirst few rows:")
display(df.head())


Total duplicate records: 13579
Unique image hashes: 6370

First few rows:


,image_path,image_name,team,class,extension,image_hash,size_mb,width,height,device,metadata,corrupted
0,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,F-89.jpg,AI-4o,frene,.jpg,80174b1e924bfd5d,2.54,3120,3120,Xiaomi,"{""544"": 0, ""545"": 0, ""546"": 0, ""547"": 0, ""548""...",no
1,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,F471.jpg,RUSTICUS,frenes,.jpg,80174b1e924bfd5d,2.54,3120,3120,Xiaomi,"{""544"": 0, ""545"": 0, ""546"": 0, ""547"": 0, ""548""...",no
2,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG_20240625_161920.jpg,AI-4o,frene,.jpg,801b37dc18ad7d56,0.06,250,250,unknown,{},no
3,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG_20240625_161920(1).jpg,AI-4o,frene,.jpg,801b37dc18ad7d56,0.06,250,250,unknown,{},no
4,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,newimages_00941.jpg,the Neural Ninjas,chene,.jpg,802965a66bd83f5b,0.86,2500,2500,SONY NEX-7,"{""PrintImageMatching"": ""PrintIM\u00000300\u000...",no


## 3. Analyze Duplicate Groups

In [16]:
# Analyze duplicate group sizes
group_sizes = df.groupby('image_hash').size()
print(f"Duplicate group distribution:")
print(group_sizes.value_counts().sort_index())

# Show example of largest duplicate group
largest_group_hash = group_sizes.idxmax()
largest_group_size = group_sizes.max()
print(f"\nLargest duplicate group ({largest_group_size} images):")
display(df[df['image_hash'] == largest_group_hash])


Duplicate group distribution:
2    5658
3     598
4     105
5       6
6       2
7       1
Name: count, dtype: int64

Largest duplicate group (7 images):


,image_path,image_name,team,class,extension,image_hash,size_mb,width,height,device,metadata,corrupted
6113,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG20240624161504_BURST006.jpg,condimenteum,tipu,.jpg,bbbb95d589a10d84,1.68,3468,3468,OPPO OPPO Reno5,"{""ImageWidth"": 3468, ""ImageLength"": 3468, ""Res...",no
6114,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG20240624161504_BURST003.jpg,condimenteum,tipu,.jpg,bbbb95d589a10d84,1.67,3468,3468,OPPO OPPO Reno5,"{""ImageWidth"": 3468, ""ImageLength"": 3468, ""Res...",no
6115,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG20240624161504_BURST002.jpg,condimenteum,tipu,.jpg,bbbb95d589a10d84,1.66,3468,3468,OPPO OPPO Reno5,"{""ImageWidth"": 3468, ""ImageLength"": 3468, ""Res...",no
6116,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG20240624161504_BURST005.jpg,condimenteum,tipu,.jpg,bbbb95d589a10d84,1.66,3468,3468,OPPO OPPO Reno5,"{""ImageWidth"": 3468, ""ImageLength"": 3468, ""Res...",no
6117,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG20240624161504_BURST001.jpg,condimenteum,tipu,.jpg,bbbb95d589a10d84,1.68,3468,3468,OPPO OPPO Reno5,"{""ImageWidth"": 3468, ""ImageLength"": 3468, ""Res...",no
6118,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG20240624161504_BURST000_COVER.jpg,condimenteum,tipu,.jpg,bbbb95d589a10d84,1.64,3468,3468,OPPO OPPO Reno5,"{""ImageWidth"": 3468, ""ImageLength"": 3468, ""Res...",no
6119,./Agrichallenge_dataset/AgrI_Dataset/Teams_spa...,IMG20240624161504_BURST004.jpg,condimenteum,tipu,.jpg,bbbb95d589a10d84,1.63,3468,3468,OPPO OPPO Reno5,"{""ImageWidth"": 3468, ""ImageLength"": 3468, ""Res...",no


## 4. Configure Destination Folder

In [17]:
# Set destination folder for duplicate images
destination_folder = "./Agrichallenge_dataset/Duplicates/"

# Enable/disable debug mode
debug_mode = False #input("Enable debug mode? (y/n): ").lower().startswith('y')

if debug_mode:
    print(f"🐛 DEBUG MODE ENABLED - No files will be moved, only simulation")
else:
    # Create destination folder if it doesn't exist
    os.makedirs(destination_folder, exist_ok=True)
    print(f"Destination folder '{destination_folder}' ready.")

Destination folder './Agrichallenge_dataset/Duplicates/' ready.


## 5. Apply Duplicate Removal Rules

In [18]:
# List to store images to remove
images_to_remove = []
keep_reasons = []  # Track why each image was kept

# Group by image_hash
for hash_value, group in df.groupby('image_hash'):
    if len(group) == 1:
        continue  # No duplicates for this hash
    
    reason = ""
    
    # Apply selection rules in order
    # 1. Largest size_mb
    max_size = group['size_mb'].max()
    candidates = group[group['size_mb'] == max_size]
    reason = f"largest size ({max_size} MB)"
    
    if len(candidates) == 1:
        keep = candidates.index[0]
    else:
        # 2. Highest resolution (width × height)
        candidates = candidates.copy()  # Create explicit copy to avoid warning
        candidates['resolution'] = candidates['width'] * candidates['height']
        max_resolution = candidates['resolution'].max()
        candidates = candidates[candidates['resolution'] == max_resolution]
        reason += f" + highest resolution ({max_resolution} pixels)"
        
        if len(candidates) == 1:
            keep = candidates.index[0]
        else:
            # 3. Device not equal to "unknown"
            non_unknown = candidates[candidates['device'] != 'unknown']
            if len(non_unknown) > 0:
                candidates = non_unknown
                reason += " + known device"
            
            if len(candidates) == 1:
                keep = candidates.index[0]
            else:
                # 4. Team not equal to "AI-4o"
                non_ai4o = candidates[candidates['team'] != 'AI-4o']
                if len(non_ai4o) > 0:
                    keep = non_ai4o.index[0]  # Take first if multiple
                    reason += " + non-AI-4o team"
                else:
                    keep = candidates.index[0]  # Take first if all are AI-4o
                    reason += " + first in list (all AI-4o)"
    
    # Record the reason for keeping this image
    keep_reasons.append((df.loc[keep, 'image_path'], reason))
    
    # Add all others in this group to removal list
    for idx in group.index:
        if idx != keep:
            images_to_remove.append(group.loc[idx, 'image_path'])

print(f"Analysis complete!")
print(f"Images to keep: {len(keep_reasons)}")
print(f"Images to remove: {len(images_to_remove)}")

Analysis complete!
Images to keep: 6370
Images to remove: 7209


## 6. Preview Keep/Remove Decisions

In [19]:
# Show first few keep decisions
print("Sample of images being kept and reasons:")
for i, (img_path, reason) in enumerate(keep_reasons[:5]):
    print(f"{i+1}. {os.path.basename(img_path)} - kept for: {reason}")

if len(keep_reasons) > 5:
    print(f"... and {len(keep_reasons) - 5} more")

print(f"\nFirst few images to be removed:")
for i, img_path in enumerate(images_to_remove[:10]):
    print(f"{i+1}. {img_path}")
    
if len(images_to_remove) > 10:
    print(f"... and {len(images_to_remove) - 10} more")

Sample of images being kept and reasons:
1. F471.jpg - kept for: largest size (2.54 MB) + highest resolution (9734400 pixels) + known device + non-AI-4o team
2. IMG_20240625_161920.jpg - kept for: largest size (0.06 MB) + highest resolution (62500 pixels) + first in list (all AI-4o)
3. newimages_00941.jpg - kept for: largest size (0.86 MB)
4. IMG_6241.JPG - kept for: largest size (0.03 MB) + highest resolution (62500 pixels) + first in list (all AI-4o)
5. 2024-06-25 19.21.02.jpg - kept for: largest size (2.85 MB)
... and 6365 more

First few images to be removed:
1. ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/frene/F-89.jpg
2. ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/frene/IMG_20240625_161920(1).jpg
3. ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/chene/newimages_00941.jpg
4. ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/tipu/IMG_6241(1).JPG
5. ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/frene/2024-06-25 19.21.02.jpg
6. ./Agrichalle

## 7. Move Duplicate Images

In [12]:
# Move files and track results
moved_count = 0
failed_moves = []

if debug_mode:
    print(f"🐛 DEBUG MODE: Simulating move of {len(images_to_remove)} duplicate images to '{destination_folder}'...\n")
else:
    print(f"Moving {len(images_to_remove)} duplicate images to '{destination_folder}'...\n")

for i, img_path in enumerate(images_to_remove, 1):
    try:
        # Get just the filename for destination
        filename = os.path.basename(img_path)
        dest_path = os.path.join(destination_folder, filename)
        
        # Handle filename conflicts by adding number suffix
        counter = 1
        base_name, ext = os.path.splitext(filename)
        original_dest_path = dest_path
        
        while os.path.exists(dest_path) and not debug_mode:
            new_filename = f"{base_name}_{counter}{ext}"
            dest_path = os.path.join(destination_folder, new_filename)
            counter += 1
        
        if debug_mode:
            # In debug mode, just simulate the move
            if counter > 1:
                print(f"[DEBUG] Would move: {img_path} -> {dest_path} (renamed due to conflict)")
            else:
                print(f"[DEBUG] Would move: {img_path} -> {dest_path}")
            moved_count += 1
        else:
            # Actually move the file
            shutil.move(img_path, dest_path)
            moved_count += 1
        
        # Print progress every 50 files (or every 10 in debug mode)
        progress_interval = 10 if debug_mode else 50
        if i % progress_interval == 0 or i == len(images_to_remove):
            print(f"Progress: {i}/{len(images_to_remove)} files {'simulated' if debug_mode else 'processed'}")
        
    except Exception as e:
        if debug_mode:
            print(f"[DEBUG] Would fail to move {img_path}: {e}")
        else:
            print(f"Failed to move {img_path}: {e}")
        failed_moves.append(img_path)

if debug_mode:
    print(f"\n🐛 DEBUG simulation complete!")
else:
    print(f"\n✅ Operation complete!")

🐛 DEBUG MODE: Simulating move of 7209 duplicate images to './Agrichallenge_dataset/Duplicates/'...

[DEBUG] Would move: ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/frene/F-89.jpg -> ./Agrichallenge_dataset/Duplicates/F-89.jpg
[DEBUG] Would move: ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/frene/IMG_20240625_161920(1).jpg -> ./Agrichallenge_dataset/Duplicates/IMG_20240625_161920(1).jpg
[DEBUG] Would move: ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/chene/newimages_00941.jpg -> ./Agrichallenge_dataset/Duplicates/newimages_00941.jpg
[DEBUG] Would move: ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/tipu/IMG_6241(1).JPG -> ./Agrichallenge_dataset/Duplicates/IMG_6241(1).JPG
[DEBUG] Would move: ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/frene/2024-06-25 19.21.02.jpg -> ./Agrichallenge_dataset/Duplicates/2024-06-25 19.21.02.jpg
[DEBUG] Would move: ./Agrichallenge_dataset/AgrI_Dataset/Teams_space/AI-4o/frene/2024_06_25_10_11_IMG_3171.JPG -> 

## 8. Summary and Results

In [13]:
# Display final summary
mode_text = "DEBUG SIMULATION" if debug_mode else "FINAL SUMMARY"
print(f"📊 {mode_text}")
print(f"="*50)
print(f"Total duplicate records analyzed: {len(df)}")
print(f"Unique images kept in dataset: {len(keep_reasons)}")

if debug_mode:
    print(f"Duplicate images that would be moved to '{destination_folder}': {moved_count}")
    print(f"Simulated failed moves: {len(failed_moves)}")
else:
    print(f"Duplicate images moved to '{destination_folder}': {moved_count}")
    print(f"Failed moves: {len(failed_moves)}")

if failed_moves:
    if debug_mode:
        print(f"\n⚠️ Files that would fail to move:")
        for img_path in failed_moves[:5]:
            print(f"   - {img_path}")
        if len(failed_moves) > 5:
            print(f"   ... and {len(failed_moves) - 5} more")
    else:
        print(f"\n⚠️ Failed moves saved to 'failed_moves.txt'")
        with open('failed_moves.txt', 'w') as f:
            for img_path in failed_moves:
                f.write(img_path + '\n')

# Calculate space potentially saved
removed_df = df[df['image_path'].isin(images_to_remove)]
space_saved = removed_df['size_mb'].sum()
print(f"\n💾 Estimated space {'that would be saved' if debug_mode else 'saved'}: {space_saved:.2f} MB")

if debug_mode:
    print(f"\n🐛 Debug simulation complete! Run again with debug_mode=False to actually move files.")
else:
    print(f"\n🎉 Dataset cleaning complete! Your dataset now contains only the best quality images.")

📊 DEBUG SIMULATION
Total duplicate records analyzed: 13579
Unique images kept in dataset: 6370
Duplicate images that would be moved to './Agrichallenge_dataset/Duplicates/': 7209
Simulated failed moves: 0

💾 Estimated space that would be saved: 9810.15 MB

🐛 Debug simulation complete! Run again with debug_mode=False to actually move files.
